# 05 — Kiểm tra checkpoint chính thức (T019/T021)

**Vai trò:** kiểm tra best.pt sau training chính thức; không huấn luyện lại, không đánh giá test.

In [1]:
# Vai trò: định vị project, run và preflight; không chạy training.
import os
import sys
import csv
import json
import hashlib
from copy import deepcopy
from datetime import datetime
from pathlib import Path

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/cifar_compare/train.py").is_file()), None)
assert root is not None, "Mở notebook trong thư mục project."
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

RUN_NAME = "convnext_tiny_seed42_20261117_191514_964199"
assert Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in {"", ".", ".."}
log_dir = root / "artifacts/logs" / RUN_NAME
checkpoint_path = root / "artifacts/checkpoints" / RUN_NAME / "best.pt"
config = json.loads((log_dir / "config.json").read_text(encoding="utf-8-sig"))
summary = json.loads((log_dir / "summary.json").read_text(encoding="utf-8-sig"))
with (log_dir / "metrics.csv").open(encoding="utf-8-sig") as file:
    rows = list(csv.DictReader(file))
best_row = max(rows, key=lambda r: (float(r["val_accuracy"]), -float(r["val_loss"])))
assert config["run_name"] == RUN_NAME
assert summary["model"] == config["model"]
assert int(best_row["epoch"]) == summary["best_epoch"]
assert float(best_row["val_accuracy"]) == summary["best_val_accuracy"]
assert float(best_row["val_loss"]) == summary["best_val_loss"]
assert checkpoint_path.is_file()
assert (root / summary["best_checkpoint"]).resolve() == checkpoint_path.resolve()

candidates = []
for path in (root / "reports").glob("phase3_preflight_*.json"):
    record = json.loads(path.read_text(encoding="utf-8-sig"))
    if record.get("planned_runs", {}).get(config["model"]) == RUN_NAME:
        candidates.append((path, record))
assert len(candidates) == 1, "Cần đúng một preflight ứng với run này."
preflight_path, preflight = candidates[0]
assert preflight["preflight_passed"]
def text_sha256(path):
    return hashlib.sha256(Path(path).read_text(encoding="utf-8-sig").encode("utf-8")).hexdigest()
for relative, expected_hash in preflight["source_fingerprints"].items():
    assert text_sha256(root / relative) == expected_hash, f"Nguồn đã đổi: {relative}"
assert text_sha256(root / config["split_path"]) == preflight["split_sha256"]
for key, value in preflight["config"].items():
    if key not in {"model", "run_name"}:
        assert config[key] == value, f"Config khác preflight: {key}"
assert Path(sys.executable).resolve() == (root / ".venv-1/Scripts/python.exe").resolve(), \
    "Hãy chọn kernel .venv-1."
print("Run:", RUN_NAME, "| model:", config["model"])
print("Best epoch:", summary["best_epoch"], "| validation accuracy:", summary["best_val_accuracy"])
print("Input/batch/workers:", config["input_size"], config["batch_size"], config["num_workers"])
print("PASS — log, config, split và fingerprints khớp.")

Run: convnext_tiny_seed42_20261117_191514_964199 | model: convnext_tiny
Best epoch: 10 | validation accuracy: 0.985
Input/batch/workers: 224 32 4
PASS — log, config, split và fingerprints khớp.


## 1. Nạp checkpoint đúng kiến trúc

Dựng model với pretrained=False để không tải weights mới. Kiểm tra metadata và strict load. safe_globals cho phép kiểu metadata TorchVersion được project lưu trong checkpoint của chính bạn.

In [2]:
# Vai trò: kiểm tra metadata và nạp trọng số; không training hoặc tải pretrained.
import torch
import torchvision
from torch.torch_version import TorchVersion
from src.cifar_compare.models import build_model, count_parameters
from src.cifar_compare.train import run_epoch
from src.cifar_compare.utils import set_seed

assert torch.cuda.is_available(), "CUDA chưa sẵn sàng."
device = torch.device("cuda")
with torch.serialization.safe_globals([TorchVersion]):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
assert checkpoint["model_name"] == config["model"]
assert checkpoint["num_classes"] == config["num_classes"] == 10
assert checkpoint["epoch"] == summary["best_epoch"]
assert checkpoint["config"] == config
assert checkpoint["weights"] == config["weights"] == "IMAGENET1K_V1"
assert checkpoint["class_names"] == preflight["class_names"]
assert checkpoint["optimizer_state_dict"]["state"], "Thiếu optimizer state."
assert checkpoint["scheduler_state_dict"] is not None
assert str(checkpoint["torch_version"]) == str(torch.__version__), "Phiên bản torch đã đổi."
assert str(checkpoint["torchvision_version"]) == str(torchvision.__version__), \
    "Phiên bản torchvision đã đổi."
assert checkpoint["best_val_accuracy"] == summary["best_val_accuracy"]
assert checkpoint["best_val_loss"] == summary["best_val_loss"]
assert all(torch.isfinite(t).all().item() for t in checkpoint["model_state_dict"].values()), \
    "Weights có NaN/Inf."

set_seed(int(config["seed"]), bool(config["deterministic"]))
model = build_model(
    model_name=checkpoint["model_name"], num_classes=checkpoint["num_classes"],
    pretrained=False, weights_name=config["weights"],
    freeze_backbone=bool(config["freeze_backbone"]),
)
load_result = model.load_state_dict(checkpoint["model_state_dict"], strict=True)
assert not load_result.missing_keys and not load_result.unexpected_keys
assert count_parameters(model) == config["total_parameters"]
model = model.to(device).eval()
print("PASS — strict load, weights hữu hạn, metadata khớp.")
print("GPU:", torch.cuda.get_device_name(device), "| Tham số:", count_parameters(model))

PASS — strict load, weights hữu hạn, metadata khớp.
GPU: NVIDIA GeForce RTX 5070 Ti | Tham số: 27827818


## 2. Forward một batch validation

Tạo loaders theo config của run. Helper chung tạo cả ba loader, nhưng chỉ duyệt validation; train/test chỉ kiểm tra số mẫu. Kiểm tra dữ liệu local trước để tránh tự tải lại.

In [3]:
# Vai trò: kiểm tra batch và logits; không backward/optimizer step.
from torchvision.datasets import CIFAR10
from torch.utils.data import SequentialSampler
from src.cifar_compare.data import get_dataloaders

offline_data = CIFAR10(root=config["data_root"], train=True, download=False)
del offline_data
offline_test = CIFAR10(root=config["data_root"], train=False, download=False)
del offline_test
reload_config = deepcopy(config)
train_loader, val_loader, test_loader, class_names = get_dataloaders(reload_config)
assert len(train_loader.dataset) == 45000
assert len(val_loader.dataset) == 5000
assert len(test_loader.dataset) == 10000
assert class_names == checkpoint["class_names"]
assert isinstance(val_loader.sampler, SequentialSampler) and not val_loader.drop_last
assert val_loader.batch_size == config["batch_size"]
assert val_loader.num_workers == config["num_workers"]
images, labels = next(iter(val_loader))
assert tuple(images.shape) == (config["batch_size"], 3, config["input_size"], config["input_size"])
assert images.dtype == torch.float32 and labels.dtype == torch.int64
assert torch.isfinite(images).all().item()
assert labels.min().item() >= 0 and labels.max().item() < 10
with torch.inference_mode():
    logits = model(images.to(device))
assert tuple(logits.shape) == (len(labels), 10)
assert torch.isfinite(logits).all().item()
print("PASS — forward validation FP32:", tuple(logits.shape), "| finite=True")
del train_loader, test_loader, images, labels, logits

PASS — forward validation FP32: (32, 10) | finite=True


## 3. Tính lại validation và lưu kết quả

Duyệt đủ 5.000 ảnh validation bằng run_epoch với optimizer=None và cùng AMP trong log gốc. Accuracy yêu cầu khớp số ảnh đúng; loss cho phép abs_tol=1e-4, rel_tol=1e-3.

In [4]:
# Vai trò: xác minh validation sau reload, lưu bằng chứng; không huấn luyện.
import math

amp_enabled = bool(config["amp"] and device.type == "cuda")
assert amp_enabled == bool(config["amp_enabled"])
validation = run_epoch(model, val_loader, device, optimizer=None, amp_enabled=amp_enabled)
expected_accuracy = float(best_row["val_accuracy"])
expected_loss = float(best_row["val_loss"])
checks = {
    "strict_load": not load_result.missing_keys and not load_result.unexpected_keys,
    "validation_samples_5000": validation["samples"] == 5000,
    "metrics_finite": all(math.isfinite(validation[k]) for k in ("loss", "accuracy")),
    "accuracy_matches": math.isclose(validation["accuracy"], expected_accuracy, rel_tol=0.0, abs_tol=1e-8),
    "loss_matches": math.isclose(validation["loss"], expected_loss, rel_tol=1e-3, abs_tol=1e-4),
}
report = {
    "checked_at": datetime.now().astimezone().isoformat(),
    "run_name": RUN_NAME, "model": config["model"],
    "checkpoint": str(checkpoint_path.relative_to(root)), "epoch": checkpoint["epoch"],
    "device": str(device), "gpu": torch.cuda.get_device_name(device),
    "amp_enabled": amp_enabled, "class_names": class_names,
    "expected_validation": {"accuracy": expected_accuracy, "loss": expected_loss},
    "reloaded_validation": validation,
    "accuracy_difference": validation["accuracy"] - expected_accuracy,
    "loss_difference": validation["loss"] - expected_loss,
    "checks": checks, "passed": all(checks.values()),
    "training_performed": False, "test_evaluated": False,
}
stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
report_path = root / "reports" / f"checkpoint_reload_{RUN_NAME}_{stamp}.json"
with report_path.open("x", encoding="utf-8") as file:
    json.dump(report, file, ensure_ascii=False, indent=2)
print(json.dumps(report, ensure_ascii=False, indent=2))
print("Đã lưu:", report_path.relative_to(root))
assert report["passed"], "Chưa PASS: giữ report/output để assistant đối chiếu."
print("T019/T021 CHECKPOINT RELOAD: PASS — không đánh giá test.")

{
  "checked_at": "2026-11-17T19:50:06.056436+08:00",
  "run_name": "convnext_tiny_seed42_20261117_191514_964199",
  "model": "convnext_tiny",
  "checkpoint": "artifacts\\checkpoints\\convnext_tiny_seed42_20261117_191514_964199\\best.pt",
  "epoch": 10,
  "device": "cuda",
  "gpu": "NVIDIA GeForce RTX 5070 Ti",
  "amp_enabled": true,
  "class_names": [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
  ],
  "expected_validation": {
    "accuracy": 0.985,
    "loss": 0.058306473898887635
  },
  "reloaded_validation": {
    "loss": 0.058303282952308653,
    "accuracy": 0.985,
    "samples": 5000
  },
  "accuracy_difference": 0.0,
  "loss_difference": -3.1909465789820013e-06,
  "checks": {
    "strict_load": true,
    "validation_samples_5000": true,
    "metrics_finite": true,
    "accuracy_matches": true,
    "loss_matches": true
  },
  "passed": true,
  "training_performed": false,
  "test_evaluated": false